## Setup: Create the Test Dataset

In [1]:
import pandas as pd
import numpy as np

# Create a synthetic Company Sales Dataset
data = {
    'Transaction_ID': range(1, 11),
    'Product_Category': ['Electronics', 'Home', 'Electronics', 'Sports', 'Home', 
                         'Electronics', 'Home', 'Sports', 'Electronics', 'Electronics'],
    'Sales_Amount': [150, 200, 155, 300, 210, 180, 205, 1000, 190, 160], # 1000 is an Outlier
    'Customer_Age': [25, 34, np.nan, 45, 23, 31, 29, np.nan, 38, 40],    # Contains Nulls (NaN)
    'Rating': [5, 4, 3, 5, 2, 4, 5, 2, 4, 3]
}
df_test = pd.DataFrame(data)

# Save to CSV for practice loading files
df_test.to_csv('company_sales_test.csv', index=False)
print("Test dataset created successfully!")
df_test.head(10)

Test dataset created successfully!


,Transaction_ID,Product_Category,Sales_Amount,Customer_Age,Rating
0,1,Electronics,150,25.0,5
1,2,Home,200,34.0,4
2,3,Electronics,155,NaN,3
3,4,Sports,300,45.0,5
4,5,Home,210,23.0,2
5,6,Electronics,180,31.0,4
6,7,Home,205,29.0,5
7,8,Sports,1000,NaN,2
8,9,Electronics,190,38.0,4
9,10,Electronics,160,40.0,3


## Assignment 1: The `automated_stat_analyzer` Function

**Scenario:** A retail company needs a utility to quickly summarize sales data. This function identifies the "Central Tendency" and "Dispersion" of any numerical column.

**Requirements:**
- Accept a Pandas DataFrame and a column name.
- Calculate the Mean, Median, and Standard Deviation.
- Identify if the data is "Skewed" by comparing the Mean and Median.
- Bonus: If the column is categorical, return the Mode instead.

In [2]:
def automated_stat_analyzer(df, column_name):
    """
    Company Task: Provide a summary report of a specific data variable.

    Instructions:
    1. Check if the column is numerical or categorical.
    2. For numerical: Calculate Mean, Median, and Standard Deviation.
    3. For categorical: Calculate the Mode.
    4. Return a dictionary with these statistical measures.
    """
    if column_name not in df.columns:
        raise ValueError(f"Column \'{column_name}\' not found in DataFrame.")

    col = df[column_name]
    result = {"column": column_name}

    if pd.api.types.is_numeric_dtype(col):
        mean_val = col.mean()
        median_val = col.median()
        std_val = col.std()

        # Skew heuristic: compare mean vs median relative to std dev
        if std_val and abs(mean_val - median_val) > 0.5 * std_val:
            skew_status = "Right-Skewed (Mean > Median)" if mean_val > median_val \
                else "Left-Skewed (Mean < Median)"
        else:
            skew_status = "Approximately Symmetric"

        result.update({
            "type": "numerical",
            "mean": round(mean_val, 2),
            "median": round(median_val, 2),
            "std_dev": round(std_val, 2),
            "skew_status": skew_status,
            "missing_values": int(col.isnull().sum())
        })
    else:
        mode_val = col.mode()
        result.update({
            "type": "categorical",
            "mode": mode_val.tolist() if not mode_val.empty else None,
            "unique_values": int(col.nunique()),
            "missing_values": int(col.isnull().sum())
        })

    return result

### Test: Numerical column (`Sales_Amount`)

In [3]:
automated_stat_analyzer(df_test, "Sales_Amount")

{'column': 'Sales_Amount',
 'type': 'numerical',
 'mean': np.float64(275.0),
 'median': np.float64(195.0),
 'std_dev': np.float64(258.31),
 'skew_status': 'Approximately Symmetric',
 'missing_values': 0}

### Test: Categorical column (`Product_Category`)

In [4]:
automated_stat_analyzer(df_test, "Product_Category")

{'column': 'Product_Category',
 'type': 'categorical',
 'mode': ['Electronics'],
 'unique_values': 3,
 'missing_values': 0}

### Test: Numerical column with missing values (`Customer_Age`)

In [5]:
automated_stat_analyzer(df_test, "Customer_Age")

{'column': 'Customer_Age',
 'type': 'numerical',
 'mean': np.float64(33.12),
 'median': np.float64(32.5),
 'std_dev': np.float64(7.59),
 'skew_status': 'Approximately Symmetric',
 'missing_values': 2}

## Assignment 2: The `null_handling_strategy` Function

**Scenario:** Incoming user data often has missing values. This function implements a flexible strategy to handle these "Null Values" to prepare data for Machine Learning.

**Requirements:**
- Check for null values in the DataFrame.
- Apply a strategy based on parameters: `"drop_rows"`, `"fill_mean"`, or `"fill_median"`.
- Ensure the function only fills numerical columns when using mean or median.

In [6]:
def null_handling_strategy(df, strategy="fill_mean"):
    """
    Company Task: Clean a dataset by resolving missing (NaN) values.

    strategy options:
      - "drop_rows": drop any row containing a null value
      - "fill_mean": fill nulls in numerical columns with column mean
      - "fill_median": fill nulls in numerical columns with column median
    """
    valid_strategies = {"drop_rows", "fill_mean", "fill_median"}
    if strategy not in valid_strategies:
        raise ValueError(f"strategy must be one of {valid_strategies}")

    df_clean = df.copy()
    null_counts_before = df_clean.isnull().sum()

    if strategy == "drop_rows":
        df_clean = df_clean.dropna()
    else:
        numeric_cols = df_clean.select_dtypes(include=np.number).columns
        for col in numeric_cols:
            if df_clean[col].isnull().any():
                if strategy == "fill_mean":
                    fill_value = df_clean[col].mean()
                else:  # fill_median
                    fill_value = df_clean[col].median()
                df_clean[col] = df_clean[col].fillna(fill_value)

    null_counts_after = df_clean.isnull().sum()

    print("Nulls before:\n", null_counts_before[null_counts_before > 0])
    print("\nNulls after:\n", null_counts_after[null_counts_after > 0] if null_counts_after.sum() > 0 else "None")

    return df_clean

### Test: `fill_mean` strategy (on `Customer_Age`)

In [7]:
df_mean_filled = null_handling_strategy(df_test, strategy="fill_mean")
df_mean_filled[["Transaction_ID", "Customer_Age"]]

Nulls before:
 Customer_Age    2
dtype: int64

Nulls after:
 None


,Transaction_ID,Customer_Age
0,1,25.000
1,2,34.000
2,3,33.125
3,4,45.000
4,5,23.000
5,6,31.000
6,7,29.000
7,8,33.125
8,9,38.000
9,10,40.000


### Test: `fill_median` strategy (on `Customer_Age`)

In [8]:
df_median_filled = null_handling_strategy(df_test, strategy="fill_median")
df_median_filled[["Transaction_ID", "Customer_Age"]]

Nulls before:
 Customer_Age    2
dtype: int64

Nulls after:
 None


,Transaction_ID,Customer_Age
0,1,25.0
1,2,34.0
2,3,32.5
3,4,45.0
4,5,23.0
5,6,31.0
6,7,29.0
7,8,32.5
8,9,38.0
9,10,40.0


### Test: `drop_rows` strategy

In [9]:
df_dropped = null_handling_strategy(df_test, strategy="drop_rows")
print(f"Rows before: {len(df_test)}, Rows after drop: {len(df_dropped)}")
df_dropped[["Transaction_ID", "Customer_Age"]]

Nulls before:
 Customer_Age    2
dtype: int64

Nulls after:
 None
Rows before: 10, Rows after drop: 8


,Transaction_ID,Customer_Age
0,1,25.0
1,2,34.0
3,4,45.0
4,5,23.0
5,6,31.0
6,7,29.0
8,9,38.0
9,10,40.0
